# 0. Output Parser Overview (the concept + the family)

Before the individual parsers, let's understand **what an output parser is**, **why it exists**, the
**`BaseOutputParser` family**, and the **shared interface** every parser follows. Learn this once and
every later file becomes easy.

---

## 1. Simple Definition

> **Kid version:** The AI talks like a person — it gives you a whole sentence. But your program is
> picky: it wants a clean list, or a number, or a filled form. An **output parser** is a little
> **translator** that stands at the exit door, takes the AI's sentence, and hands your program exactly
> the tidy thing it asked for.

**Professional definition:** An *output parser* is a component that transforms an LLM's raw output
(usually the text of an `AIMessage`) into a structured Python value — a `str`, `list`, `dict`,
`datetime`, `Enum`, or a validated object. All parsers inherit from **`BaseOutputParser`** and share a
common interface.

```python
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
parser.parse("apple, banana, cherry")   # ['apple', 'banana', 'cherry']
```

---

## 2. Why Do Output Parsers Exist?

**The problem:** Models output **text**. Programs need **data types**. Bridging that gap by hand
(regex, `str.split`, `json.loads`) is repetitive and fragile, and it has to be redone for every
project.

### Before output parsers

```python
resp = model.invoke("List 3 fruits").content    # "1. Apple\n2. Banana\n3. Cherry"
fruits = [l.split(". ")[1] for l in resp.splitlines()]   # breaks if format changes
```

### After output parsers

```python
parser = CommaSeparatedListOutputParser()
chain = prompt.partial(format_instructions=parser.get_format_instructions()) | model | parser
chain.invoke({})     # ['apple', 'banana', 'cherry']  — reliable + reusable
```

Two wins bundled into one object:
1. **`get_format_instructions()`** tells the model *how* to format (added to the prompt).
2. **`parse()`** converts the reply into the right Python type.

Plus, parsers are **Runnables**, so they snap onto the end of any chain with `|`.

---

## 3. Real-Life Analogy

A **translator + quality inspector at a factory exit** 🏭. Raw product (the model's text) comes off the
line. The inspector (parser) checks it, reshapes it into the standard package your warehouse expects
(list/dict/object), and rejects anything malformed. Nothing leaves the building in a shape your
systems can't handle.

---

## 4. Where Output Parsers Fit (the family tree)

```
Runnable
   │  (gives .invoke() and the |  pipe operator)
   ▼
BaseOutputParser
   │   defines: parse(), get_format_instructions(), invoke(), parse_result()
   │
   ├── StrOutputParser                         → str          
   ├── BaseTransformOutputParser               (adds streaming support)
   │       └── ...list/json parsers stream through here
   ├── CommaSeparatedListOutputParser          → list[str]    
   ├── StructuredOutputParser                  → dict         
   ├── JsonOutputParser                        → dict (stream)
   ├── PydanticOutputParser                    → BaseModel    
   ├── DatetimeOutputParser                    → datetime     
   ├── EnumOutputParser                        → Enum         
   └── OutputFixingParser / RetryOutputParser  → wrap another parser
```

- Every parser is a **`BaseOutputParser`**, so they all share the same methods.
- Streaming-capable parsers descend from **`BaseTransformOutputParser`** /
  **`BaseCumulativeTransformOutputParser`**.
- Because they're **Runnables**, the standard place for a parser is the **end of a chain**:
  `prompt | model | parser`.

---

## 5. Internal Working (the two phases)

```
  ┌──────────── PHASE 1: BEFORE the model (prompt building) ───────────────┐
  │  parser.get_format_instructions()                                      │
  │     → "Return a comma-separated list" / "Return JSON {...}" / ...      │
  │     → injected into the prompt (usually via partial_variables)         │
  └────────────────────────────────────────────────────────────────────────┘
                                   │
                                   ▼
                            model generates text
                                   │
  ┌──────────── PHASE 2: AFTER the model (parsing) ────────────────────────┐
  │  parser.parse(text)  (or parse_result on the raw generation)           │
  │     → clean the text (strip fences/prose)                              │
  │     → convert to the target type (list / dict / object / ...)          │
  │     → (some parsers) validate; on failure raise OutputParserException  │
  └────────────────────────────────────────────────────────────────────────┘
                                   │
                                   ▼
                        clean Python value to your code
```

When used in a chain via `.invoke()`, LangChain calls `parse` (or `parse_result`) for you
automatically — you just place the parser after the model.

---

## 6. The Shared Interface (every parser has these)

### `parse(text)`

**Definition:** converts raw text into the type/structure that the parser is designed to produce.

**What does parse(text) actually mean?**

Consider:

parser.parse("apple, banana")

There are two important things here:

parser → an object that knows how to interpret the text

.parse() → the method that performs that interpretation

"apple, banana" → the raw text being given to the parser

So:
```
"apple, banana"
       ↓
    parse()
       ↓
["apple", "banana"]
```
The parser looks at the raw text and applies some rules to transform it.
```
Raw format                 Target format

"apple, banana"    ───→    ["apple", "banana"]
```

The parser acts like a translator between two representations.

For example:
```
String
  ↓
parse()
  ↓
List
```

Or:
```
JSON string
  ↓
parse()
  ↓
Python dictionary
```

Or:
```
LLM text
  ↓
parse()
  ↓
Pydantic object
```

**Why can't Python automatically do this?**

Suppose you have:

text = "apple, banana"

Python sees this simply as:

str

It doesn't automatically know that you mean:

["apple", "banana"]

Maybe you wanted:

"apple, banana"

to remain a string.

Or maybe you wanted:

("apple", "banana")

Or:

{"first": "apple", "second": "banana"}

Python needs a rule.

That's what the parser provides.

**The parser contains the rules:**

Imagine we create a parser whose rule is:

"Whenever you receive comma-separated text, split it at commas."

Conceptually:
```
class MyParser:
    def parse(self, text):
        return text.split(",")
```
Now:

parser = MyParser()

result = parser.parse("apple, banana")

Python performs:

"apple, banana".split(",")

which gives:

["apple", " banana"]

We could additionally remove whitespace:
```
class MyParser:
    def parse(self, text):
        return [item.strip() for item in text.split(",")]
```
Now:

parser.parse("apple, banana")

returns:

["apple", "banana"]

This is the essence of parsing.

The parser is not communicating with the LLM.

The LLM has already produced the text.

The parser receives that text and processes it locally according to its rules.

So:
```
LLM
 │
 │ generates
 ▼
"apple, banana"
 │
 │ given to parser
 ▼
parse()
 │
 │ transforms
 ▼
["apple", "banana"]
```

**Why it exists:** It's the core transformation.

**Real-life use case:** The inspector reshaping raw product into a standard package.

```python
parser.parse("apple, banana")   # -> ['apple', 'banana']
```

---

### `get_format_instructions()`

**Definition:** Returns text describing the required output format, to embed in the prompt.

**Why it exists:** The model must be *told* the format so its reply is parseable.

**When developers use it:** Inject into the prompt, usually via `partial_variables`.

```python
prompt = PromptTemplate(
    template="{query}\n{format_instructions}",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)
```

> `StrOutputParser` doesn't really need this (any text is fine).

---

### `invoke() / the `|` operator`

**Definition:** Because parsers are Runnables, `.invoke(text_or_message)` runs the parse, and `|`
chains the parser after a model.

**Why it exists:** Uniformity — parsers compose exactly like prompts and models.

```python
chain = prompt | model | parser
chain.invoke({"query": "..."})
```

---

### `parse_result()` *(advanced)*

**Definition:** Parses the full list of `Generation` objects from a model (not just a string) — lets
parsers access richer info (e.g. function-call args, multiple candidates).

**Why it exists:** Some parsers need more than the plain text (e.g. OpenAI tools parsers).

---

### Streaming (for transform parsers)

**Definition:** Streaming-capable parsers (`StrOutputParser`, `JsonOutputParser`) can emit partial
results as tokens arrive via `.stream()`.

**Why it exists:** Live UIs that show the answer building up.

```python
for piece in (prompt | model | JsonOutputParser()).stream({...}):
    print(piece)   # growing partial dict
```

---

## 7. The typical chain shape

```python
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
prompt = PromptTemplate(
    template="List 5 {things}.\n{format_instructions}",
    input_variables=["things"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)
chain = prompt | ChatOpenAI(model="gpt-4o-mini") | parser
chain.invoke({"things": "fruits"})     # ['apple', 'banana', 'cherry', 'date', 'elderberry']
```

Memorize this shape — **every** parser file uses the same `prompt | model | parser` pattern.
